# QuRe Exp1 · Gaussian query tokens qua prompt flow

Mỗi truy vấn có 10 mẫu độc lập khi train. Mỗi mẫu là **một bộ 32 token, mỗi token 768 chiều**. Ở thời điểm khởi tạo, từng tọa độ của bộ token đầu vào flow có phân phối Gaussian với mean 0 và độ lệch chuẩn lấy từ `Qformer.config.initializer_range`. Các vị trí token có mean và log-scale học được. Embedding trung bình của caption (bỏ padding) điều kiện hóa Gaussian và tham số flow. Mỗi bước planar biến đổi trực tiếp vector 768 chiều.

Mỗi bộ token sau flow đi vào Q-Former cùng caption và visual memory của ảnh reference. Cả 10 lần dùng chung trọng số. Target dùng bộ query pretrained của Exp1. Với mỗi cặp query–target, lấy max trên 32 target token riêng cho từng mẫu rồi lấy trung bình 10 điểm. Train dùng contrastive loss trong batch. Khi validation, 10 mẫu nhiễu được cố định để so sánh được giữa các epoch và batch.

Bật GPU và Internet trên Kaggle, gắn **FashionIQ đã giải nén**, rồi chọn **Run All**. Hạ `NUM_FLOWS` xuống 0 để chạy đối chứng Gaussian cùng 10 mẫu, seed, loss và cấu hình train. Hai chế độ lưu riêng checkpoint. Nhiều mẫu làm tăng thời gian chạy Q-Former.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import shutil
import hashlib

NUM_SAMPLES = 10
NUM_FLOWS = 10
assert NUM_SAMPLES >= 1 and NUM_FLOWS >= 0
RUNS = [f'gaussian_query_s{NUM_SAMPLES}_f{NUM_FLOWS}']
DATA_ROOT = None  # Path('/kaggle/input/.../fashionIQ') nếu tự tìm không được
CHECKPOINT = None  # Checkpoint BLIP-2 pretrained, không phải checkpoint đã train
RESUME_FROM = {}  # Ví dụ {RUNS[0]: '/kaggle/input/run/last.pt'}
SEED = 1000
BATCH_SIZE = 16
EPOCHS = 30
RUN_TAG = 's1'  # Đổi tag để bắt đầu lượt mới với cấu hình khác
WORK = Path('/kaggle/working')
EXPERIMENT_SHA = 'b36607df9d7813167df9cf2ef1d1de605938f1f2'
BACKBONE_SHA = 'c1c7f6b62fe6d556b9554b6c7cae111f71f67917'
RUNNER_SHA = '2be8e012314dbfb4f46b406e6a6e633a5e8a055a'
RUNNER = WORK / 'exp1_direct_runner_source'
PACKAGE = RUNNER / 'experiments' / 'exp1_direct_query_pfm'
BACKBONE = WORK / 'qure_main_backbone'
CODE = WORK / 'exp1_gaussian_query_flow'
CODE.mkdir(exist_ok=True)
assert BATCH_SIZE >= 2


## Cài thư viện và lấy source đã khóa commit

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle settings.")
subprocess.run([sys.executable, "-m", "pip", "install", "--prefer-binary", "--only-binary=tokenizers",
                "transformers==4.46.3", "tokenizers==0.20.3",
                "timm==0.9.16", "fairscale", "omegaconf", "iopath"], check=True)

# Verify dependencies in a fresh process, as used by the training runner.
subprocess.run([sys.executable, "-c", """
import sys, torch, transformers, tokenizers
from transformers.modeling_utils import (
    PreTrainedModel, apply_chunking_to_forward,
    find_pruneable_heads_and_indices, prune_linear_layer,
)
assert transformers.__version__ == "4.46.3"
assert tokenizers.__version__ == "0.20.3"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__, "| tokenizers:", tokenizers.__version__)
"""], check=True)

def checkout(repo, sha, destination):
    if not (destination / ".git").is_dir():
        destination.mkdir(exist_ok=True)
        subprocess.run(["git", "init", "-q", str(destination)], check=True)
    probe = subprocess.run(["git", "-C", str(destination), "rev-parse", "--verify", "HEAD"], capture_output=True, text=True)
    if probe.stdout.strip() != sha:
        subprocess.run(["git", "-C", str(destination), "fetch", "--depth=1", repo, sha], check=True)
        subprocess.run(["git", "-C", str(destination), "checkout", "--detach", sha], check=True)
    print(destination.name, subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip())

checkout("https://github.com/fast2104/QuRe.git", RUNNER_SHA, RUNNER)
checkout("https://github.com/fast2104/QuRe.git", BACKBONE_SHA, BACKBONE)

## Dataset và checkpoint pretrained

In [ ]:
import shutil
inputs = Path("/kaggle/input")
if DATA_ROOT is None:
    matches = [p.parent.parent for p in inputs.rglob("cap.dress.train.json")
               if (p.parent.parent / "images").is_dir() and (p.parent.parent / "image_splits").is_dir()]
    if len(matches) != 1:
        raise ValueError(f"Set DATA_ROOT to the extracted FashionIQ directory. Candidates: {matches}")
    DATA_ROOT = matches[0]
DATA_ROOT = Path(DATA_ROOT)
for category in ("dress", "shirt", "toptee"):
    for split in ("train", "val"):
        annotations = DATA_ROOT / "captions" / f"cap.{category}.{split}.json"
        sample = json.loads(annotations.read_text())[0]
        for key in ("candidate", "target"):
            image = DATA_ROOT / "images" / (sample[key] + ".png")
            if not image.is_file():
                raise FileNotFoundError(image)
    assert (DATA_ROOT / "image_splits" / f"split.{category}.val.json").is_file()

name = "blip2_pretrained_vitL.pth"
if CHECKPOINT is None:
    CHECKPOINT = next(inputs.rglob(name), WORK / name)
CHECKPOINT = Path(CHECKPOINT).resolve()
if not CHECKPOINT.is_file():
    if CHECKPOINT != WORK / name:
        raise FileNotFoundError(CHECKPOINT)
    base_url = "https://storage.googleapis.com/sfr-vision-language-research/LAVIS/models/BLIP2/"
    temporary = WORK / (name + ".part")
    subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(temporary), base_url + name], check=True)
    temporary.replace(CHECKPOINT)
if CHECKPOINT.name != name:
    raise ValueError(f"Expected checkpoint filename {name}")
os.environ["QURe_CHECKPOINT_ROOT"] = str(CHECKPOINT.parent)
print("FashionIQ:", DATA_ROOT, "| BLIP-2:", CHECKPOINT)

## Chuẩn bị backbone

In [ ]:
def original(rel):
    return subprocess.check_output(["git", "-C", str(BACKBONE), "show", f"HEAD:{rel}"], text=True)

model_init = "lavis/models/__init__.py"
source = original(model_init)
loaders = source[source.index("def load_model("):source.index("class ModelZoo")]
imports = """import logging
import torch
from omegaconf import OmegaConf
from lavis.common.registry import registry
from lavis.models.base_model import BaseModel
from lavis.processors import BaseProcessor
from lavis.models.blip2_models.blip2_qure import Blip2QuRe
"""
(BACKBONE / model_init).write_text(imports + "\n" + loaders)
(BACKBONE / "lavis/processors/__init__.py").write_text(
    "from lavis.processors.base_processor import BaseProcessor\nfrom lavis.processors import blip_processors\n")
(BACKBONE / "data/__init__.py").write_text("# Retrieval runner imports only data.utils.\n")
yaml_path = "lavis/configs/models/blip2/blip2_pretrain_vitL.yaml"
import re
yaml_text, count = re.subn(r"(?m)^  pretrained:.*$", lambda _: "  pretrained: " + json.dumps(str(CHECKPOINT)), original(yaml_path))
assert count == 1
(BACKBONE / yaml_path).write_text(yaml_text)

## Gaussian query tokens và planar flow
Mã PyTorch của biến thể nằm trong cell dưới. Notebook sử dụng model, dữ liệu và preflight GPU từ commit của Exp1.

In [ ]:
for filename in ("model.py", "data.py", "preflight.py"):
    shutil.copyfile(PACKAGE / filename, CODE / filename)
module_source = r'''"""Sample reference query tokens in Q-Former space, then transform each with planar flow."""
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

from model import PositiveQure


class GaussianQueryFlow(nn.Module):
    def __init__(self, slots, width, init_std, *, samples, steps, context_width, eval_seed):
        super().__init__()
        if samples < 1 or steps < 0 or init_std <= 0:
            raise ValueError("Require samples >= 1, steps >= 0 and init_std > 0")
        self.samples = samples
        self.steps = steps
        self.width = width
        self.init_std = init_std
        self.slot_mean = nn.Parameter(torch.zeros(slots, width))
        self.slot_log_std = nn.Parameter(torch.zeros(slots, width))
        self.text_context = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, context_width), nn.SiLU())
        self.gaussian = nn.Linear(context_width, 2 * width)
        nn.init.zeros_(self.gaussian.weight)
        nn.init.zeros_(self.gaussian.bias)
        if steps:
            self.planar = nn.Linear(context_width, steps * (2 * width + 1))
        generator = torch.Generator(device="cpu").manual_seed(eval_seed)
        self.register_buffer("eval_eps", torch.randn(samples, slots, width, generator=generator))

    def forward(self, caption_embedding):
        with torch.autocast(device_type=caption_embedding.device.type, enabled=False):
            c = self.text_context(caption_embedding.float())
            mean_offset, log_scale = self.gaussian(c).chunk(2, -1)
            mean = self.slot_mean + mean_offset[:, None]
            std = (self.slot_log_std + log_scale[:, None]).clamp(-5, 3).exp()
            shape = (len(c), self.samples, *self.slot_mean.shape)
            eps = torch.randn(shape, device=c.device) if self.training else self.eval_eps[None]
            # Work in units of the original initializer std for numerical conditioning.
            start = mean[:, None] + std[:, None] * eps
            z = start
            if self.steps:
                params = self.planar(c).reshape(len(c), self.steps, 2 * self.width + 1)
                for step in params.unbind(1):
                    u, w, bias = step.split([self.width, self.width, 1], -1)
                    w = F.normalize(w, dim=-1)
                    dot = (u * w).sum(-1, keepdim=True)
                    u = u + (F.softplus(dot) - 1 - dot) * w
                    h = torch.tanh((z * w[:, None, None]).sum(-1, keepdim=True) + bias[:, None, None])
                    z = z + u[:, None, None] * h
            return start * self.init_std, z * self.init_std


def sample_spread(banks):
    """RMS of per-coordinate standard deviation across samples of the same query."""
    return banks.detach().float().var(dim=1, unbiased=False).mean().sqrt()


class PromptFlowQure(PositiveQure):
    def __init__(self, *args, pfm_config, **kwargs):
        super().__init__(*args, **kwargs)
        if self.experiment != "exp1" or self.use_variance:
            raise ValueError("This notebook implements the standard Exp1 in-batch objective")
        with torch.random.fork_rng(devices=[]):
            self.ref_flow = GaussianQueryFlow(
                self.num_query_tokens, self.hidden_size, self.base.Qformer.config.initializer_range,
                samples=pfm_config["samples"], steps=pfm_config["num_flows"],
                context_width=pfm_config["hidden"], eval_seed=pfm_config["eval_seed"])

    def _encode_bank(self, bank, visual, ids, attention_mask):
        hidden = self._qformer(query_embeds=bank, input_ids=ids, attention_mask=attention_mask,
                               encoder_hidden_states=visual, encoder_attention_mask=self._memory_mask(visual))
        projected = self.base.text_proj(hidden[:, :self.num_query_tokens])
        return F.normalize(projected.mean(dim=1), dim=-1)

    def _sample_features(self, images, captions):
        visual = self._vision(images)
        text = self._tokenize(captions)
        words = self.base.Qformer.bert.embeddings.word_embeddings(text.input_ids)
        mask = text.attention_mask.unsqueeze(-1)
        pooled = (words * mask).sum(1) / mask.sum(1).clamp_min(1)
        start, banks = self.ref_flow(pooled)
        query_mask = text.attention_mask.new_ones((len(images), self.num_query_tokens))
        attention = torch.cat([query_mask, text.attention_mask], dim=1)
        features = []
        for bank in banks.unbind(dim=1):
            args = (bank, visual, text.input_ids, attention)
            if torch.is_grad_enabled():
                feature = checkpoint(self._encode_bank, *args, use_reentrant=False)
            else:
                feature = self._encode_bank(*args)
            features.append(feature)
        return torch.stack(features, dim=1), start, banks

    def query_features(self, reference_images, captions):
        features, _, _ = self._sample_features(reference_images, captions)
        return features

    @staticmethod
    def sample_scores(features, targets):
        # Each sample sees all target tokens. Average only after the QuRe max.
        return torch.einsum("bsd,jkd->bsjk", features, targets).max(dim=-1).values.mean(1)

    def forward(self, reference_images, target_images, captions, target_names):
        features, start, banks = self._sample_features(reference_images, captions)
        targets = self.target_features(target_images)
        logits = self.sample_scores(features, targets) / self.base.temp
        mask = self._positive_mask(target_names, logits.device)
        positive = torch.logsumexp(logits.masked_fill(~mask, -torch.inf), dim=1)
        loss = (torch.logsumexp(logits, dim=1) - positive).mean()
        return {"loss": loss, "loss_contrastive": loss, "loss_var": loss.new_zeros(()),
                "pre_flow_spread": sample_spread(start), "post_flow_spread": sample_spread(banks),
                "feature_spread": sample_spread(features)}

    @torch.no_grad()
    def score(self, query_features, gallery_features, chunk_size=256):
        rows = []
        for queries in query_features.split(32):
            scores = [self.sample_scores(queries, targets.to(queries.device)).cpu()
                      for targets in gallery_features.split(chunk_size)]
            rows.append(torch.cat(scores, dim=1))
        return torch.cat(rows)
'''
(CODE / 'pfm_variants.py').write_text(module_source, encoding='utf-8')

In [ ]:
trainer = r'''"""Train/evaluate positive-only QuRe experiments with QuRe baseline settings."""

from __future__ import annotations

import argparse
import importlib.util
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm

from model import Exp5TextAuxQure, PairwiseKLQure, PositiveQure, VICRegQure, load_qure
from pfm_variants import PromptFlowQure


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True, type=Path)
    parser.add_argument("--qure-root", required=True, type=Path)
    parser.add_argument("--fashioniq-root", required=True, type=Path)
    parser.add_argument("--resume", type=Path)
    parser.add_argument("--output-dir", type=Path)
    return parser.parse_args()


def load_local_data_module():
    """Load our data.py without registering it as ``data``.

    QuRe itself imports ``data.utils``.  Giving this local module a distinct
    name prevents it from shadowing that original QuRe package.
    """
    path = Path(__file__).with_name("data.py")
    spec = importlib.util.spec_from_file_location("positive_fashioniq_data", path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cosine_lr(optimizer, epoch, max_epochs, init_lr, min_lr):
    lr = (init_lr - min_lr) * 0.5 * (1.0 + math.cos(math.pi * epoch / max_epochs)) + min_lr
    for group in optimizer.param_groups:
        group["lr"] = lr
    return lr


def make_epoch_scheduler(optimizer, init_lr, min_lr, max_epochs):
    """The same per-epoch cosine schedule used by Code-ablation."""
    min_ratio = min_lr / init_lr

    def cosine_factor(epoch):
        progress = min(max(epoch, 0) / max(max_epochs, 1), 1.0)
        return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=cosine_factor)


def rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].detach().cpu())
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all([item.detach().cpu() for item in state["cuda"]])


def parameter_summary(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return {
        "total_parameters": total,
        "trainable_parameters": trainable,
        "frozen_parameters": total - trainable,
        "trainable_parameter_percent": 100.0 * trainable / total if total else 0.0,
    }


@torch.no_grad()
def evaluate(model, validation_loaders, text_processor, device):
    model.eval()
    metrics = {}
    recalls10, recalls50 = [], []
    spread_sums = dict(pre_flow_spread=0., post_flow_spread=0., feature_spread=0.)
    num_queries = 0
    for dress_type, loaders in validation_loaders.items():
        gallery_features, gallery_names = [], []
        for images, names in tqdm(loaders["gallery"], desc=f"{dress_type}: gallery", leave=False):
            features = model.target_features(images.to(device, non_blocking=True))
            gallery_features.append(features.cpu())
            gallery_names.extend(names)
        gallery_features = torch.cat(gallery_features)

        query_features, target_names = [], []
        for references, captions, targets in tqdm(loaders["queries"], desc=f"{dress_type}: queries", leave=False):
            captions = [text_processor(caption) for caption in captions]
            features, start, banks = model._sample_features(references.to(device, non_blocking=True), captions)
            for key, tensor in zip(spread_sums, (start, banks, features)):
                squared_spread = tensor.float().var(dim=1, unbiased=False).mean()
                spread_sums[key] += squared_spread.item() * len(features)
            num_queries += len(features)
            del start, banks
            query_features.append(features.cpu())
            target_names.extend(targets)
        query_features = torch.cat(query_features).to(device)

        scores = model.score(query_features, gallery_features)
        sorted_indices = scores.argsort(dim=-1, descending=True).numpy()
        names = np.asarray(gallery_names)[sorted_indices]
        labels = names == np.asarray(target_names)[:, None]
        if not np.all(labels.sum(axis=1) == 1):
            raise RuntimeError(f"Expected exactly one FashionIQ target in {dress_type} gallery")
        # Keep the same FashionIQ benchmark fields used by Code-ablation.
        recall10 = float(labels[:, :10].any(axis=1).mean() * 100.0)
        recall50 = float(labels[:, :50].any(axis=1).mean() * 100.0)
        metrics[f"fashionIQ_{dress_type}_recall@10"] = recall10
        metrics[f"fashionIQ_{dress_type}_recall@50"] = recall50
        recalls10.append(recall10)
        recalls50.append(recall50)
    metrics["average_recall_at10"] = float(np.mean(recalls10))
    metrics["average_recall_at50"] = float(np.mean(recalls50))
    metrics["average_recall"] = (
        metrics["average_recall_at10"] + metrics["average_recall_at50"]
    ) / 2.0
    metrics.update({f"val_{key}": math.sqrt(total / num_queries) for key, total in spread_sums.items()})
    return metrics


def save_checkpoint(model, optimizer, scheduler, scaler, epoch, global_step,
                    best_metric, best_epoch, config, metrics):
    """Checkpoint payload compatible in structure with Code-ablation."""
    return {
        "format_version": 1,
        "checkpoint_type": "positive_qure_full",
        "epoch": epoch,
        "global_step": global_step,
        "best_metric": best_metric,
        "best_epoch": best_epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "rng_state": rng_state(),
        "configs": config,
        "metrics": metrics,
    }


def atomic_save_checkpoint(path, **kwargs):
    path = Path(path)
    temporary_path = path.with_name(f".{path.name}.tmp")
    try:
        torch.save(save_checkpoint(**kwargs), temporary_path)
        os.replace(temporary_path, path)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise


def load_training_checkpoint(path, model, optimizer, scheduler, scaler):
    try:
        state = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:  # PyTorch versions before the weights_only argument.
        state = torch.load(path, map_location="cpu")
    model.load_state_dict(
        state["model_state_dict"] if "model_state_dict" in state else state["model"]
    )
    optimizer.load_state_dict(
        state["optimizer_state_dict"] if "optimizer_state_dict" in state else state["optimizer"]
    )
    if "scheduler_state_dict" in state:
        scheduler.load_state_dict(state["scheduler_state_dict"])
    if "scaler_state_dict" in state:
        scaler.load_state_dict(state["scaler_state_dict"])
    restore_rng_state(state.get("rng_state"))
    return state


def main():
    args = parse_args()
    config = json.loads(args.config.read_text(encoding="utf-8"))
    output_dir = args.output_dir or Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    seed_everything(config["seed"])
    run_started = time.time()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("This QuRe/BLIP-2 training implementation requires CUDA.")

    data_module = load_local_data_module()
    base, vis_processors, txt_processors = load_qure(config["backbone"], device, args.qure_root)
    if config["num_query_tokens"] != base.query_tokens.shape[1]:
        raise ValueError(
            "Config num_query_tokens must match the loaded QuRe checkpoint "
            f"({config['num_query_tokens']} != {base.query_tokens.shape[1]})."
        )
    if config["max_txt_len"] != base.max_txt_len:
        raise ValueError(
            "Config max_txt_len must match the loaded QuRe checkpoint "
            f"({config['max_txt_len']} != {base.max_txt_len})."
        )
    loss_name = config.get("loss", "inbatch_contrastive")
    if loss_name == "inbatch_contrastive":
        model_class = PositiveQure
        main_loss_key = "loss_contrastive"
        loss_metric_keys = ["loss", main_loss_key, "loss_var"]
    elif loss_name == "pairwise_kl_preference":
        model_class = PairwiseKLQure
        main_loss_key = "loss_pairwise_kl"
        loss_metric_keys = ["loss", main_loss_key, "loss_var"]
    elif loss_name == "exp1_text_aux_contrastive":
        model_class = Exp5TextAuxQure
        main_loss_key = "loss_contrastive_reference_text"
        loss_metric_keys = [
            "loss", "loss_contrastive_reference_text", "loss_contrastive_text", "loss_var"
        ]
    elif loss_name == "vicreg":
        model_class = VICRegQure
        main_loss_key = "loss_align"
        loss_metric_keys = ["loss", "loss_align", "loss_var", "loss_cov"]
    else:
        raise ValueError(
            "Unknown loss. Expected 'inbatch_contrastive', "
            "'pairwise_kl_preference', 'exp1_text_aux_contrastive', or 'vicreg'; "
            f"got: {loss_name!r}"
        )
    model_args = (
        base, config["experiment"], config["max_txt_len"], config["use_variance"],
        config["lambda_var"], config["var_gamma"], config["var_epsilon"],
    )
    if loss_name == "exp1_text_aux_contrastive":
        model = model_class(*model_args, lambda_text=config["lambda_text"]).to(device)
    elif loss_name == "vicreg":
        model = model_class(
            *model_args,
            lambda_align=config["lambda_align"],
            lambda_v=config["lambda_v"],
            lambda_c=config["lambda_c"],
            projector_hidden_dim=config["vicreg_projector_hidden_dim"],
            projector_output_dim=config["vicreg_projector_output_dim"],
            queue_size=config["vicreg_queue_size"],
        ).to(device)
    elif config["pfm"]["variant"] != "baseline":
        model = PromptFlowQure(*model_args, pfm_config=config["pfm"]).to(device)
        loss_metric_keys += ["pre_flow_spread", "post_flow_spread", "feature_spread"]
    else:
        model = model_class(*model_args).to(device)
    parameters = parameter_summary(model)
    print(
        "[MODEL] "
        f"total={parameters['total_parameters']:,} "
        f"trainable={parameters['trainable_parameters']:,} "
        f"frozen={parameters['frozen_parameters']:,} "
        f"({parameters['trainable_parameter_percent']:.3f}% trainable)",
        flush=True,
    )

    # QuRe FashionIQDataset itself uses targetpad_transform(1.25, 224), not
    # LAVIS's train transform. Reuse exactly that transform implementation.
    import sys
    if str(args.qure_root.resolve()) not in sys.path:
        sys.path.insert(0, str(args.qure_root.resolve()))
    from data.utils import targetpad_transform
    transform = targetpad_transform(config["target_ratio"], config["image_size"])
    train_loader = data_module.make_train_loader(
        args.fashioniq_root, transform, config["batch_size"], config["num_workers"]
    )
    val_loaders = data_module.make_validation_loaders(
        args.fashioniq_root, transform, config["eval_batch_size"], config["num_workers"]
    )

    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=config["init_lr"], weight_decay=config["weight_decay"],
    )
    scheduler = make_epoch_scheduler(
        optimizer, config["init_lr"], config["min_lr"], config["epochs"]
    )
    scaler = torch.cuda.amp.GradScaler(enabled=config["amp"])
    start_epoch, best_score, best_epoch, global_step = 0, -float("inf"), None, 0
    history_path = output_dir / "history.json"
    history = []
    if history_path.exists():
        try:
            history = json.loads(history_path.read_text(encoding="utf-8"))
            if not isinstance(history, list):
                history = []
        except (OSError, json.JSONDecodeError):
            history = []
    if args.resume:
        state = load_training_checkpoint(args.resume, model, optimizer, scheduler, scaler)
        start_epoch = state["epoch"] + 1
        global_step = state.get("global_step", start_epoch * len(train_loader))
        best_score = state.get("best_metric", state.get("metrics", {}).get("average_recall", best_score))
        best_epoch = state.get("best_epoch")

    for epoch in range(start_epoch, config["epochs"]):
        model.epoch = epoch
        model.train()
        lr = optimizer.param_groups[0]["lr"]
        running = {key: 0.0 for key in loss_metric_keys}
        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{config['epochs']}")
        for step, (references, targets, captions, target_names) in enumerate(progress, start=1):
            captions = [txt_processors["eval"](caption) for caption in captions]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=config["amp"]):
                losses = model(
                    references.to(device, non_blocking=True),
                    targets.to(device, non_blocking=True), captions, target_names,
                )
            if not torch.isfinite(losses["loss"]):
                raise FloatingPointError(f"Non-finite loss at epoch {epoch + 1}, step {step}")
            scaler.scale(losses["loss"]).backward()
            scaler.step(optimizer)
            scaler.update()
            if loss_name == "vicreg":
                model.update_vicreg_queue()
            global_step += 1
            for key in running:
                running[key] += float(losses[key].detach())
            progress.set_postfix(loss=running["loss"] / step, lr=lr)

        scheduler.step()
        lr = optimizer.param_groups[0]["lr"]
        epoch_metrics = {key: value / len(train_loader) for key, value in running.items()}
        epoch_metrics.update({"lr": lr, "epoch": epoch + 1, "global_step": global_step})
        loss_report = " ".join(
            f"{key}={epoch_metrics[key]:.6f}" for key in loss_metric_keys
        )
        print(f"[EPOCH {epoch + 1}/{config['epochs']}] {loss_report} lr={lr:.3e}", flush=True)
        if (epoch + 1) % config["evaluate_every"] == 0:
            epoch_metrics.update(evaluate(model, val_loaders, txt_processors["eval"], device))
            print(
                f"[VALIDATION EPOCH {epoch + 1}]\n"
                + json.dumps(
                    {key: value for key, value in epoch_metrics.items() if "recall" in key},
                    indent=2,
                    sort_keys=True,
                ),
                flush=True,
            )
            model.train()
        history.append({"epoch": epoch + 1, **epoch_metrics})
        selection_metric = epoch_metrics.get("average_recall", -epoch_metrics["loss"])
        is_best = selection_metric > best_score
        if is_best:
            best_score, best_epoch = selection_metric, epoch + 1
        checkpoint_kwargs = dict(
            model=model, optimizer=optimizer, scheduler=scheduler, scaler=scaler,
            epoch=epoch, global_step=global_step, best_metric=best_score,
            best_epoch=best_epoch, config=config, metrics=epoch_metrics,
        )
        atomic_save_checkpoint(output_dir / "last.pt", **checkpoint_kwargs)
        if is_best:
            atomic_save_checkpoint(output_dir / "best.pt", **checkpoint_kwargs)
        (output_dir / "latest_metrics.json").write_text(
            json.dumps(epoch_metrics, indent=2), encoding="utf-8"
        )
        history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")
        (output_dir / "run_summary.json").write_text(json.dumps({
            "run_id": output_dir.name,
            "experiment_dir": str(output_dir.resolve()),
            "config": config,
            "parameters": parameters,
            "optimizer": "AdamW",
            "scheduler": "cosine_per_epoch",
            "best_epoch": best_epoch,
            "best_metric": best_score,
            "best_checkpoint": str((output_dir / "best.pt").resolve()),
            "last_epoch": epoch + 1,
            "global_step": global_step,
            "elapsed_seconds": time.time() - run_started,
            "history_file": str(history_path.resolve()),
        }, indent=2), encoding="utf-8")


if __name__ == "__main__":
    main()
'''
(CODE / 'train_pfm.py').write_text(trainer, encoding='utf-8')

## Kiểm tra trước khi train
Kiểm tra shape, gradient, phép gộp điểm và đối chứng 0 flow trên tensor nhỏ. Preflight ở cell huấn luyện kiểm tra thêm với backbone và ảnh FashionIQ thật.

In [ ]:
unit_source = r'''"""CPU checks of query sampling, joint reference encoding and the retrieval score."""
from types import SimpleNamespace

import torch
from torch import nn
from pfm_variants import GaussianQueryFlow, PromptFlowQure, sample_spread


def check():
    torch.manual_seed(1000)
    flow = GaussianQueryFlow(32, 768, 0.02, samples=10, steps=4, context_width=64, eval_seed=101).eval()
    context = torch.randn(2, 768)
    with torch.no_grad():
        start, banks = flow(context)
        assert banks.shape == (2, 10, 32, 768)
        torch.testing.assert_close(start[0], 0.02 * flow.eval_eps)
        torch.testing.assert_close(banks[:1], flow(context[:1])[1], atol=2e-6, rtol=1e-4)
        assert sample_spread(banks) > 0 and not torch.equal(banks, start)
        assert not torch.equal(banks[0], banks[1])
        print(f"Eval on synthetic captions: Gaussian spread={sample_spread(start):.6f}, flow spread={sample_spread(banks):.6f}")
        flow.train()
        assert not torch.equal(flow(context)[1], flow(context)[1])
        control = GaussianQueryFlow(32, 768, 0.02, samples=10, steps=0, context_width=64, eval_seed=101).eval()
        a, b = control(context)
        torch.testing.assert_close(a, b)

    class ToyJoint(PromptFlowQure):
        def _tokenize(self, captions):
            return SimpleNamespace(input_ids=captions, attention_mask=torch.ones_like(captions))

        def _vision(self, images):
            return images

        def _qformer(self, *, query_embeds, encoder_hidden_states, input_ids=None, **kwargs):
            fused = query_embeds + encoder_hidden_states.mean(1, keepdim=True)
            if input_ids is not None:
                fused = fused + self.base.Qformer.bert.embeddings.word_embeddings(input_ids).mean(1, keepdim=True)
            return fused.tanh()

    base = nn.Module()
    base.query_tokens = nn.Parameter(torch.randn(1, 3, 8))
    base.Qformer = nn.Module()
    base.Qformer.config = SimpleNamespace(initializer_range=0.02, hidden_size=8, encoder_width=8)
    base.Qformer.bert = nn.Module()
    base.Qformer.bert.embeddings = nn.Module()
    base.Qformer.bert.embeddings.word_embeddings = nn.Embedding(20, 8)
    base.text_proj, base.vision_proj = nn.Linear(8, 5), nn.Linear(8, 5)
    base.temp = nn.Parameter(torch.tensor(0.07))
    model = ToyJoint(base, "exp1", 4, False, 0, 1, 1e-4,
                     pfm_config=dict(samples=10, num_flows=4, hidden=6, eval_seed=101))
    refs, target_images = torch.randn(3, 5, 8), torch.randn(3, 5, 8)
    ids = torch.randint(0, 20, (3, 4))
    model.eval()
    q = model.query_features(refs, ids)
    t = model.target_features(target_images)
    assert q.shape == (3, 10, 5) and t.shape == (3, 3, 5)
    assert torch.autograd.grad(q.sum(), base.query_tokens, allow_unused=True)[0] is None
    expected = torch.stack([torch.einsum("bd,jkd->bjk", item, t).max(-1).values for item in q.unbind(1)]).mean(0)
    torch.testing.assert_close(model.score(q, t, 2), expected)
    model.train()
    loss = model(refs, target_images, ids, ["a", "a", "b"])["loss"]
    loss.backward()
    for parameter in (model.ref_flow.gaussian.weight, model.ref_flow.planar.weight,
                      model.ref_flow.slot_mean, model.ref_flow.slot_log_std, base.query_tokens):
        assert torch.isfinite(parameter.grad).all() and parameter.grad.abs().sum() > 0
    print("PASS Gaussian control, fixed eval, stochastic train, score aggregation and gradients.")


if __name__ == "__main__":
    check()
'''
(CODE / 'verify_variants.py').write_text(unit_source, encoding='utf-8')
subprocess.run([sys.executable, str(CODE / "verify_variants.py")], cwd=CODE, check=True)

## Huấn luyện
Checkpoint last.pt lưu mỗi epoch và best.pt theo Avg validation tốt nhất. Resume cần checkpoint cùng phiên bản/cấu hình. Với session mới, gắn outputs cũ vào Kaggle Input và khai báo RESUME_FROM. Cuối buổi dùng Save Version với outputs.

In [ ]:
template = json.loads((PACKAGE / 'config.json').read_text())
config_paths = {}
output_paths = {}
source_digest = hashlib.sha256((module_source + trainer).encode()).hexdigest()
for variant in RUNS:
    output = WORK / f'exp1_suite_{variant}_seed{SEED}_{RUN_TAG}'
    pfm = dict(variant=variant, hidden=64, num_flows=NUM_FLOWS,
               samples=NUM_SAMPLES, eval_seed=SEED + 1)
    config = dict(template, seed=SEED, batch_size=BATCH_SIZE, epochs=EPOCHS,
                  output_dir=str(output), pfm=pfm, source_sha256=source_digest,
                  experiment_sha=EXPERIMENT_SHA, backbone_sha=BACKBONE_SHA)
    config_path = CODE / f'config_{variant}.json'
    config_path.write_text(json.dumps(config, indent=2))
    config_paths[variant] = config_path
    output_paths[variant] = output
    resume = Path(RESUME_FROM[variant]) if variant in RESUME_FROM else output / 'last.pt'
    if variant in RESUME_FROM and not resume.is_file():
        raise FileNotFoundError(resume)
    if resume.is_file():
        saved = json.loads((resume.parent / 'config.json').read_text())
        expected = dict(config)
        saved.pop('output_dir', None)
        expected.pop('output_dir', None)
        if saved != expected:
            raise ValueError(f'Checkpoint/config mismatch: {resume}. Use matching settings or a new RUN_TAG.')
        output.mkdir(exist_ok=True)
        if resume.parent.resolve() != output.resolve():
            for filename in ('history.json', 'run_summary.json', 'latest_metrics.json', 'best.pt'):
                source = resume.parent / filename
                if source.exists():
                    shutil.copy2(source, output / filename)
    elif (output / 'history.json').exists():
        raise RuntimeError(f'History exists without last.pt in {output}; select a new RUN_TAG.')
    subprocess.run([sys.executable, str(CODE / 'preflight.py'), str(config_path),
                    str(BACKBONE), str(DATA_ROOT)], cwd=CODE, check=True)
    output.mkdir(exist_ok=True)
    for filename in ('pfm_variants.py', 'train_pfm.py', 'model.py', 'data.py'):
        shutil.copy2(CODE / filename, output / filename)
    command = [sys.executable, '-u', str(CODE / 'train_pfm.py'),
               '--config', str(config_path), '--qure-root', str(BACKBONE),
               '--fashioniq-root', str(DATA_ROOT), '--output-dir', str(output)]
    if resume.is_file():
        command += ['--resume', str(resume)]
    with (output / 'console.log').open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=CODE, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)


## Recall và độ phân tán trên validation
Với cùng một cặp ảnh–caption, đo phương sai giữa 10 mẫu theo từng tọa độ, lấy trung bình rồi căn bậc hai. `val_pre_flow_spread` và `val_post_flow_spread` đo cùng đơn vị query token. `val_feature_spread` đo riêng trên feature retrieval đã chuẩn hóa khi tắt dropout. Đối chiếu hai lượt 0 flow và có flow ở cùng epoch.

In [ ]:
import pandas as pd
rows = []
for variant, output in output_paths.items():
    history = json.loads((output / 'history.json').read_text())
    evaluated = [r for r in history if 'average_recall' in r]
    if evaluated:
        last = evaluated[-1]
        best = max(evaluated, key=lambda r: r['average_recall'])
        rows.append(dict(variant=variant, last_epoch=last['epoch'],
                         pre_flow=last['val_pre_flow_spread'], post_flow=last['val_post_flow_spread'],
                         feature_spread=last['val_feature_spread'], R10=last['average_recall_at10'], R50=last['average_recall_at50'],
                         Avg=last['average_recall'], best_epoch=best['epoch'], best_Avg=best['average_recall']))
summary = pd.DataFrame(rows)
display(summary.round(3))
summary.to_csv(CODE / 'results_summary.csv', index=False)
